## This notebook scrapes data about wine production per state, wine production from Sonoma, specifically, and compares this to representation of Sonoma County wine carried by Joy Wines in Denver, CO.

### Annual wine production (in gallons) per state, can be found here: https://worldpopulationreview.com/state-rankings/wine-production-by-state

In [1]:
#url = https://worldpopulationreview.com/state-rankings/wine-production-by-state
#there is a <table> element with a <thead> containing State, Annual wine production 2023 (gal) and % of national production in 2023.

In [2]:
import pandas as pd

In [3]:
wine_production_table = pd.read_html('https://worldpopulationreview.com/state-rankings/wine-production-by-state')

In [4]:
wine_prod_df = wine_production_table[0]

In [5]:
wine_prod_df_clean = wine_prod_df.drop(columns='Unnamed: 0')

In [6]:
wine_prod_df_clean.head()

,State,Annual Wine Production 2023 (Gal)↓,% of National Production 2023
0,California,609607342,80.79%
1,New York,33084702,4.38%
2,Washington,32373972,4.29%
3,Oregon,17091139,2.27%
4,Kentucky,7815921,1.04%


In [7]:
wine_prod_df_clean.to_csv('wine_production_by_state.csv')

In [8]:
#total wine production in US
total_prod = sum(wine_prod_df_clean['Annual Wine Production 2023 (Gal)↓'])
total_prod

1488000580

In [9]:
#wine prod West Coast U.S. (defining West Coast as California, Oregon, and Washington State)
west_coast_states = ['California', 'Oregon', 'Washington']
west_coast = wine_prod_df_clean['State'].isin(west_coast_states)
west_prod = sum(wine_prod_df_clean[west_coast]['Annual Wine Production 2023 (Gal)↓'])
west_prod


659072453

In [10]:
#What % of total wine production in U.S. is from west coast states:

round(west_prod/total_prod * 100)

44

#### Sonoma County has 62,000 acres of grapes
1 acre of grapes = 3958 bottles of wine
1 bottle of wine = 39 ounces
1 gallon = 128 ounces

In [11]:
#What % of west coast prod is from Sonoma, CA

sonoma_prod_ounces = 62000*3958*39
sonoma_prod_gal = sonoma_prod_ounces/128
sonoma_prod_gal

round(sonoma_prod_gal/total_prod * 100)

5

#### How much of Joy Wines West Coast wall is from Sonoma County?
For this, I hand-collected data for a snapshot of their inventory. On Tuesday June 23 2026, around 5pm, Joy Wine & Spirits had 252 unique bottles of wine on their West Coast Wall (i.e., I only counted one bottle per label). Of this, 27 were from Sonoma County wineries

In [12]:
joy_sonoma_inventory = round(27/252 * 100)

## How much small-label representation does Joy Wines have
Part of what makes Joy feel special is the density of small-label wine representation they have in-store. 

### Which Sonoma labels does Joy carry, specifically?

Joy Wines uses a JavaScript-rendered site, so direct scraping wasn't possible. Instead, I used the browser's network inspector to identify the underlying API endpoint the site was calling to populate its inventory, and saved the JSON response directly.

In [15]:
import json

with open('joy_wines_sonoma_inventory.json') as data:
    sonoma_data = json.load(data)

In [16]:
type(sonoma_data)

dict

In [17]:
sonoma_data.keys()

dict_keys(['result', 'data'])

In [18]:
sonoma_data['data'][3]['values']

[{'display': 'Sonoma Cutrer',
  'count': 2,
  'query': {'brands': 'Sonoma Cutrer'},
  'active': False},
 {'display': 'La Crema Winery',
  'count': 2,
  'query': {'brands': 'La Crema Winery'},
  'active': False},
 {'display': 'Chardonnay',
  'count': 2,
  'query': {'brands': 'Chardonnay'},
  'active': False},
 {'display': 'The Prisoner Wine Company',
  'count': 1,
  'query': {'brands': 'The Prisoner Wine Company'},
  'active': False},
 {'display': 'Penny Island',
  'count': 1,
  'query': {'brands': 'Penny Island'},
  'active': False},
 {'display': 'Josh Cellars',
  'count': 1,
  'query': {'brands': 'Josh Cellars'},
  'active': False},
 {'display': 'Cline Family Cellars',
  'count': 1,
  'query': {'brands': 'Cline Family Cellars'},
  'active': False},
 {'display': 'Bloodroot',
  'count': 1,
  'query': {'brands': 'Bloodroot'},
  'active': False},
 {'display': 'Bedrock',
  'count': 1,
  'query': {'brands': 'Bedrock'},
  'active': False}]

In [19]:
sonoma_data_df = pd.json_normalize(sonoma_data['data'][3]['values'])
sonoma_data_df

,display,count,active,query.brands
0,Sonoma Cutrer,2,False,Sonoma Cutrer
1,La Crema Winery,2,False,La Crema Winery
2,Chardonnay,2,False,Chardonnay
3,The Prisoner Wine Company,1,False,The Prisoner Wine Company
4,Penny Island,1,False,Penny Island
5,Josh Cellars,1,False,Josh Cellars
6,Cline Family Cellars,1,False,Cline Family Cellars
7,Bloodroot,1,False,Bloodroot
8,Bedrock,1,False,Bedrock


In [20]:
hand_collected = pd.DataFrame([
    {'display': 'Broc Cellars'},
    {'display': 'Ultraviolet Wines'}
])

joy_sonoma_brands = pd.concat([sonoma_data_df, hand_collected], ignore_index=True)
joy_sonoma_brands

,display,count,active,query.brands
0,Sonoma Cutrer,2.0,False,Sonoma Cutrer
1,La Crema Winery,2.0,False,La Crema Winery
2,Chardonnay,2.0,False,Chardonnay
3,The Prisoner Wine Company,1.0,False,The Prisoner Wine Company
4,Penny Island,1.0,False,Penny Island
5,Josh Cellars,1.0,False,Josh Cellars
6,Cline Family Cellars,1.0,False,Cline Family Cellars
7,Bloodroot,1.0,False,Bloodroot
8,Bedrock,1.0,False,Bedrock
9,Broc Cellars,NaN,NaN,NaN


In [21]:
joy_sonoma_brands.to_csv('joy_inventory.csv')

### Case count per label?
What is the "size" of each label carried by Joy Wines (via case count/yr)? How does this compare to the type of Sonoma Wine you can get elsewhere?

The goal is to find the median case count across Joy's Sonoma selection, and compare that to a high-production Sonoma winery and the label you'd most likely find "anywhere" (grocery stores, chain restaurants, etc.).

To answer these questions, I manually looked up the case count/yr of each brand on the joy_sonoma_brands list on https://www.wineindustrydata.com/index/lightsearch. 

Notes/process:
- Look up each brand's case count/yr

- Wine Industry Data provides case counts as a range, to simplify things, I entered the median value for each brand's case count/yr

- If I couldn't find the data on the Wine Industry Data website, I researched it via other means

- Sonoma Cutrer wasn't on the Wine Industry Data website, so I found their case count (543000 cases/yr) here: https://sonomawinegrape.org/sonoma-cutrer-chardonnay-thoroughbreds-and-croquets/ 

- I couldn't find any information online about Penny Island's annual case production, so I ommited them from the final list. The other labels had a wide range of case count (from 500 - 1000000), so I didn't feel confident filling in an estimate for Penny Island


In [22]:
# Annual case production per label, researched manually via wineindustrydata.com
# Penny Island excluded — no production data available
case_counts = {
    'Sonoma Cutrer': 543000,
    'La Crema Winery': 3750,
    'The Prisoner Wine Company': 500,
    'Josh Cellars': 1000000,
    'Cline Family Cellars': 1000000,
    'Bloodroot': 8750,
    'Bedrock': 3750,
    'Broc Cellars': 8750,
    'Ultraviolet Wines': 500
}

case_count_df = pd.DataFrame(list(case_counts.items()), columns=['brand', 'case_count_yr'])
case_count_df



,brand,case_count_yr
0,Sonoma Cutrer,543000
1,La Crema Winery,3750
2,The Prisoner Wine Company,500
3,Josh Cellars,1000000
4,Cline Family Cellars,1000000
5,Bloodroot,8750
6,Bedrock,3750
7,Broc Cellars,8750
8,Ultraviolet Wines,500


In [23]:
# Median case count across Joy's Sonoma inventory
joy_median = case_count_df['case_count_yr'].median()
joy_median

np.float64(8750.0)

Kendall-Jackson is the highest-production winery in Sonoma County and one of the best-selling wine brands in the U.S., and represents the kind of bottle of Sonoma County wine you'd like find anywhere that sells wine. It produces approximately 6 million cases per year (source: wineinternationalassociation.org), making it a useful benchmark for what mass-market Sonoma wine looks like.

source: (https://wineinternationalassociation.org/top-30-largest-wineries-in-2025-ranked-by-production-scale-and-premium-wine-quality-in-the-u-s/)

The comparison below puts Joy's median Sonoma label production against Kendall-Jackson's annual output.

In [24]:
# Kendall-Jackson (Jackson Family Wines) = ~6 million cases/yr
# Source: wine industry reporting
joy_vs_anywhere = pd.DataFrame([
    {'name': 'Joy Wines Inventory', 'case_count_yr': joy_median},
    {'name': 'Jackson Family Wines', 'case_count_yr': 6000000}
])

joy_vs_anywhere

,name,case_count_yr
0,Joy Wines Inventory,8750.0
1,Jackson Family Wines,6000000.0


In [25]:
joy_vs_anywhere.to_csv('joy_vs_anywhere.csv')